PERGUNTAS A SEREM RESPONDIDAS

1. Qual é a tendencia de volume de pedidos da Olist ao longo do tempo?
2. Como o custo médio do frete varia geograficamente pelo Brasil?
3. Qual é o impacto real e quantificável de demora na entrega sobre a nota de satisfação do cliente?
4. Quais categorias de produtos do catálog mais puxam a satisfação para baixo e sofrem com a complexidade logística?

### CARREGAMENTO DO DATASET

In [1]:
import kagglehub
import pandas as pd
import numpy as np

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

c:\Users\lugul\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## CARREGAMENTO DOS DADOS ##

df_orders = pd.read_csv("C:/dev/SCTEC_site/Trilha-Dados/kaggle/olist_orders_dataset.csv")
df_customers = pd.read_csv("C:/dev/SCTEC_site/Trilha-Dados/kaggle/olist_customers_dataset.csv")
df_reviews = pd.read_csv("C:/dev/SCTEC_site/Trilha-Dados/kaggle/olist_order_reviews_dataset.csv")
df_items = pd.read_csv("C:/dev/SCTEC_site/Trilha-Dados/kaggle/olist_order_items_dataset.csv")
df_products = pd.read_csv("C:/dev/SCTEC_site/Trilha-Dados/kaggle/olist_products_dataset.csv")

In [11]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [13]:
df_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [14]:
df_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


### LIMPEZA INICIAL E FILTROS DE INTEGRIDADE

In [16]:
df_orders = df_orders[df_orders["order_status"] == "delivered"].copy()
date_cols_ordes = ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]

for col in date_cols_ordes:
    df_orders[col] = pd.to_datetime(df_orders[col])

df_items["shipping_limit_date"] = pd.to_datetime(df_items["shipping_limit_date"])

df_reviews_clean = df_reviews.groupby("order_id", as_index=False)["review_score"].mean()

In [17]:
df_orders.info()

<class 'pandas.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  str           
 1   customer_id                    96478 non-null  str           
 2   order_status                   96478 non-null  str           
 3   order_purchase_timestamp       96478 non-null  datetime64[us]
 4   order_approved_at              96464 non-null  datetime64[us]
 5   order_delivered_carrier_date   96476 non-null  datetime64[us]
 6   order_delivered_customer_date  96470 non-null  datetime64[us]
 7   order_estimated_delivery_date  96478 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.6 MB


In [18]:
df_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [20]:
df_reviews_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 98673 entries, 0 to 98672
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      98673 non-null  str    
 1   review_score  98673 non-null  float64
dtypes: float64(1), str(1)
memory usage: 1.5 MB


In [21]:
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [22]:
df_customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### MERGE DAS TABELAS

In [24]:
df_main = df_orders.merge(df_customers[["customer_id","customer_state","customer_city"]], on="customer_id", how="left")
df_main.head()
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  str           
 1   customer_id                    96478 non-null  str           
 2   order_status                   96478 non-null  str           
 3   order_purchase_timestamp       96478 non-null  datetime64[us]
 4   order_approved_at              96464 non-null  datetime64[us]
 5   order_delivered_carrier_date   96476 non-null  datetime64[us]
 6   order_delivered_customer_date  96470 non-null  datetime64[us]
 7   order_estimated_delivery_date  96478 non-null  datetime64[us]
 8   customer_state                 96478 non-null  str           
 9   customer_city                  96478 non-null  str           
dtypes: datetime64[us](5), str(5)
memory usage: 7.4 MB


In [28]:
df_main = df_main.merge(df_items[[
"order_id",
"product_id",
"seller_id",
"price",
"freight_value",
"shipping_limit_date"
]], on="order_id", how="left")
df_main.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,product_id,seller_id,price,freight_value,shipping_limit_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,2017-10-06 11:07:15
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,2018-07-30 03:24:27
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,2018-08-13 08:55:23
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,sao goncalo do amarante,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,2017-11-23 19:45:59
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,santo andre,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,2018-02-19 20:31:37


In [29]:
df_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [31]:
df_main = df_main.merge(df_products[[
"product_id",
"product_category_name"
]], on="product_id", how="left")
df_main.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,product_id,seller_id,price,freight_value,shipping_limit_date,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,2017-10-06 11:07:15,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,2018-07-30 03:24:27,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,2018-08-13 08:55:23,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,sao goncalo do amarante,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,2017-11-23 19:45:59,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,santo andre,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,2018-02-19 20:31:37,papelaria


In [33]:
df_main = df_main.merge(df_reviews_clean, on="order_id", how="left")
df_main.head()
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 110197 entries, 0 to 110196
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110197 non-null  str           
 1   customer_id                    110197 non-null  str           
 2   order_status                   110197 non-null  str           
 3   order_purchase_timestamp       110197 non-null  datetime64[us]
 4   order_approved_at              110182 non-null  datetime64[us]
 5   order_delivered_carrier_date   110195 non-null  datetime64[us]
 6   order_delivered_customer_date  110189 non-null  datetime64[us]
 7   order_estimated_delivery_date  110197 non-null  datetime64[us]
 8   customer_state                 110197 non-null  str           
 9   customer_city                  110197 non-null  str           
 10  product_id                     110197 non-null  str           
 11  seller_id  

In [34]:
df_main.dropna(subset=["order_purchase_timestamp"],inplace=True)
print("Banco de dados consolidado.")

Banco de dados consolidado.


In [35]:
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 110197 entries, 0 to 110196
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110197 non-null  str           
 1   customer_id                    110197 non-null  str           
 2   order_status                   110197 non-null  str           
 3   order_purchase_timestamp       110197 non-null  datetime64[us]
 4   order_approved_at              110182 non-null  datetime64[us]
 5   order_delivered_carrier_date   110195 non-null  datetime64[us]
 6   order_delivered_customer_date  110189 non-null  datetime64[us]
 7   order_estimated_delivery_date  110197 non-null  datetime64[us]
 8   customer_state                 110197 non-null  str           
 9   customer_city                  110197 non-null  str           
 10  product_id                     110197 non-null  str           
 11  seller_id  

### FEATURE EMGINEERING (REGRAS DE NEGÓCIO)

In [40]:
df_main["tempo_entrega_dias"] = (df_main["order_delivered_customer_date"] - df_main["order_purchase_timestamp"]).dt.days
df_main["atraso_entrega_dias"] = (df_main["order_delivered_customer_date"] - df_main["order_estimated_delivery_date"]).dt.days

print(f"Base consolidada finalizada com sucesso! Linhas: {df_main.shape[0]}, Colunas: {df_main.shape[1]}")
df_main.head(3)

Base consolidada finalizada com sucesso! Linhas: 110197, Colunas: 20


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,customer_city,product_id,seller_id,price,freight_value,shipping_limit_date,product_category_name,review_score_x,review_score_y,tempo_entrega_dias,atraso_entrega_dias
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,sao paulo,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,2017-10-06 11:07:15,utilidades_domesticas,4.0,4.0,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,barreiras,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,2018-07-30 03:24:27,perfumaria,4.0,4.0,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,vianopolis,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,2018-08-13 08:55:23,automotivo,5.0,5.0,9.0,-18.0


In [42]:
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 110197 entries, 0 to 110196
Data columns (total 20 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110197 non-null  str           
 1   customer_id                    110197 non-null  str           
 2   order_status                   110197 non-null  str           
 3   order_purchase_timestamp       110197 non-null  datetime64[us]
 4   order_approved_at              110182 non-null  datetime64[us]
 5   order_delivered_carrier_date   110195 non-null  datetime64[us]
 6   order_delivered_customer_date  110189 non-null  datetime64[us]
 7   order_estimated_delivery_date  110197 non-null  datetime64[us]
 8   customer_state                 110197 non-null  str           
 9   customer_city                  110197 non-null  str           
 10  product_id                     110197 non-null  str           
 11  seller_id  

In [43]:
df_main["product_category_name"] = df_main["product_category_name"].fillna("Não informada")

In [44]:
df_main.info()

<class 'pandas.DataFrame'>
RangeIndex: 110197 entries, 0 to 110196
Data columns (total 20 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110197 non-null  str           
 1   customer_id                    110197 non-null  str           
 2   order_status                   110197 non-null  str           
 3   order_purchase_timestamp       110197 non-null  datetime64[us]
 4   order_approved_at              110182 non-null  datetime64[us]
 5   order_delivered_carrier_date   110195 non-null  datetime64[us]
 6   order_delivered_customer_date  110189 non-null  datetime64[us]
 7   order_estimated_delivery_date  110197 non-null  datetime64[us]
 8   customer_state                 110197 non-null  str           
 9   customer_city                  110197 non-null  str           
 10  product_id                     110197 non-null  str           
 11  seller_id  